# Log Analysis & Using pandas

This notebook analyzes the access log of an nginx web server: a sample (about one line in a hundred) of the log of an Iranian online shop, from January 22 to 26, 2019, published by Farzin Zaker on Harvard Dataverse under the CC0 license. Each line describes a request, in nginx's `combined` format (https://nginx.org/en/docs/http/ngx_http_log_module.html#log_format), followed here by one last field:

```
66.249.66.194 - - [22/Jan/2019:03:57:11 +0330] "GET /filter/b36,p3 HTTP/1.1" 200 35145 "-" "Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)" "-"
```

It holds the client's IP address, two empty fields (`-`), the date and time (in Tehran, `+0330`), the request (method, path, protocol), the response's status code, its size in bytes, the page the client comes from (the referrer, `-` if there is none), its browser (user agent), and the `X-Forwarded-For` header, always empty here.

1. Splitting a line with a regular expression
2. Loading the log into a `DataFrame`
3. Analyzing the traffic: status codes, most requested paths, traffic per hour, suspicious requests
4. A command to filter the log by date

Exercises 1 and 2 follow on from each other; the analyses of 3 use the `DataFrame` of 2 and are independent of each other; 4 depends on no other. Each exercise has a cell for your code, then additional hints and its solution, collapsed: open them if needed. The solutions define what the rest needs: if you skip an exercise, run its solution.

The next two cells download the log, then read it into the list `lines`.

In [ ]:
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/nginx-log/access-small.log

In [ ]:
import re
from pathlib import Path

import pandas as pd

lines = Path("access-small.log").read_text(encoding="utf8").splitlines()
print(f"{len(lines)} lines")
print(lines[3])

## 1. Splitting a line with a regular expression

Write a pattern `PATTERN`, compiled with `re.compile`, whose named groups split a log line into its fields: `ip`, `date`, `method`, `path`, `protocol`, `status`, `size`, `referrer` and `agent` (the last field, always empty, needs no group). The `re.VERBOSE` flag lets you write it over several lines, with comments.

Then write a function `parse_line(line)` that returns the dictionary of the pattern's groups, or `None` if the line doesn't match it. Print how the first line is split, then the number of log lines that don't match the pattern, and the first 100 characters of each: what is special about them?

To work out the pattern, https://regex101.com/ ("Python" flavor) explains each of its parts and shows what it matches in a line you paste there.

In [ ]:
# Your code here


def parse_line(line: str) -> dict[str, str] | None: ...

### Additional hints

One named group `(?P<name>...)` per field, with what the field can hold:

- `\S+` matches a run of non-whitespace characters: the IP address, the method, the path
- `[^\]]+` matches everything up to the closing bracket (the date), `[^"]*` everything up to the next double quote (the protocol, the referrer, the browser); nginx writes a double quote that is part of a field as `\x22`
- `\d{3}` and `\d+` match the status code and the size

With `re.VERBOSE`, the pattern's whitespace is ignored and `#` starts a comment: a space to match is written `\ `, preferably at the start of a line, where an editor doesn't strip it. `PATTERN.match(line)` returns `None` if the start of the line doesn't match the pattern, else a `Match` object, whose `groupdict()` gives the dictionary of the groups.

For the first line, you should find `'ip': '54.36.149.41'`, `'date': '22/Jan/2019:03:56:14 +0330'`, `'method': 'GET'`, `'status': '200'` and `'size': '30577'`. Two lines don't match: a request so long that nginx truncated and refused it (code 414), and bytes that aren't HTTP (the start of an encrypted connection, sent to the HTTP port, which nginx refuses with a 400 code).

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
PATTERN = re.compile(
  r"""
  (?P<ip>\S+)\ \S+\ \S+  # IP address, then two empty fields (-)
  \ \[(?P<date>...)\]  # Date, in brackets
  \ "(?P<method>...)\ (?P<path>...)\ (?P<protocol>...)"  # Request
  \ (?P<status>...)\ (?P<size>...)  # Status code, size
  \ "(?P<referrer>...)"\ "(?P<agent>...)"  # Referrer, browser
  """,
  re.VERBOSE,
)


def parse_line(line: str) -> dict[str, str] | None:
  match = PATTERN.match(line)
  if ...:
    return None
  return ...


print(parse_line(lines[0]))
rejected = [line for line in lines if ...]
print(f"{len(rejected)} rejected lines:")
for line in rejected:
  print(...)
```

### Solution

In [ ]:
PATTERN = re.compile(
  r"""
  (?P<ip>\S+)\ \S+\ \S+  # IP address, then two empty fields (-)
  \ \[(?P<date>[^\]]+)\]  # Date, in brackets
  \ "(?P<method>\S+)\ (?P<path>\S+)\ (?P<protocol>[^"]*)"  # Request
  \ (?P<status>\d{3})\ (?P<size>\d+)  # Status code, size
  \ "(?P<referrer>[^"]*)"\ "(?P<agent>[^"]*)"  # Referrer, browser
  """,
  re.VERBOSE,
)


def parse_line(line: str) -> dict[str, str] | None:
  match = PATTERN.match(line)
  if match is None:
    return None
  return match.groupdict()


print(parse_line(lines[0]))
rejected = [line for line in lines if parse_line(line) is None]
print(f"{len(rejected)} rejected lines:")
for line in rejected:
  print(line[:100])

## 2. Loading the log into a `DataFrame`

Build the `DataFrame` `logs`: one row per log line that `parse_line` splits, one column per group of the pattern. All its columns are then strings: convert `date` to dates with `pd.to_datetime`, then `status` and `size` to integers. Print the columns' types, the period the log covers (its first and last dates) and the first rows of `logs`.

**Useful documentation**

- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [`strptime` format codes](https://docs.python.org/3/library/datetime.html#format-codes)
- [`Series.astype`](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html)

In [ ]:
# Your code here

### Additional hints

`pd.DataFrame(dictionaries)` creates, from a list of dictionaries, one column per key. Keep the split lines only: `parse_line` returns `None` for the others. `pd.to_datetime(column, format=...)` reads the dates in the given format, with `strptime`'s codes: `%d` the day, `%b` the abbreviated month name (`Jan`), `%Y` the year, `%H:%M:%S` the time, `%z` the UTC offset (`+0330`). `column.astype(int)` converts a column of strings to integers. `min()` and `max()` give the smallest and largest values of a column.

You should find the type `datetime64[ns, UTC+03:30]` for `date`, `int64` for `status` and `size`, and a log from January 22, 2019 at 3:56 am to January 26 at 8:29 pm.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
parsed = [... for line in lines]
logs = pd.DataFrame([d for d in parsed if ...])
logs["date"] = pd.to_datetime(logs["date"], format="...")
logs["status"] = ...
logs["size"] = ...
print(logs.dtypes)
print(f"From {logs['date'].min()} to {...}")
logs.head()
```

### Solution

In [ ]:
parsed = [parse_line(line) for line in lines]
logs = pd.DataFrame([d for d in parsed if d is not None])
logs["date"] = pd.to_datetime(logs["date"], format="%d/%b/%Y:%H:%M:%S %z")
logs["status"] = logs["status"].astype(int)
logs["size"] = logs["size"].astype(int)
print(logs.dtypes)
print(f"From {logs['date'].min()} to {logs['date'].max()}")
logs.head()

## 3. Analyzing the traffic

Four analyses of `logs`, independent of each other.

### 3.1. Status codes

Count the responses of each status code. What share of the requests failed, that is, got a code of at least 400 (4xx: client error, 5xx: server error)?

In [ ]:
# Your code here

#### Additional hints

`value_counts()` counts each value of a column, from the most frequent to the least. The comparison `logs["status"] >= 400` gives a `Series` of booleans: its sum counts the `True` values, and its mean gives their proportion. In an f-string, the `:.2%` format prints a proportion as a percentage, with two decimals.

You should find 95,912 responses 200 (success), 3,379 responses 304 (not modified: the browser keeps the copy it already has), and 1,717 errors, that is 1.66% of the requests.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
print(logs["status"].value_counts())
errors = ...
print(f"{errors.sum()} errors, that is {...:.2%} of the requests")
```

#### Solution

In [ ]:
print(logs["status"].value_counts())
errors = logs["status"] >= 400
print(f"{errors.sum()} errors, that is {errors.mean():.2%} of the requests")

### 3.2. Most requested paths

Print the 10 most requested paths: they are mostly files that every page loads, such as images or fonts. Then print the 10 most requested paths among the 404 (not found) responses: they are the files to create, or the links to fix.

In [ ]:
# Your code here

#### Additional hints

`value_counts()` already sorts by decreasing frequency: `head(10)` keeps the first 10. A boolean mask, `logs["status"] == 404`, in `logs.loc[mask, "path"]`, keeps only the paths of the 404 responses.

You should find `/settings/logo` at the top (3,673 requests) and, among the 404s, icons that iPhones and iPads request when a site is added to the home screen: `/apple-touch-icon-precomposed.png` (179 requests), `/apple-touch-icon.png`…

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
print(logs["path"].value_counts().head(10))
not_found = logs.loc[..., "path"]
print(...)
```

#### Solution

In [ ]:
print(logs["path"].value_counts().head(10))
not_found = logs.loc[logs["status"] == 404, "path"]
print(not_found.value_counts().head(10))

### 3.3. Traffic per hour

Count the requests of each hour of the day (from 0 to 23, in the server's time), all days together, and plot them as a bar chart. Also compute the volume transferred each hour, in megabytes: the sum of the responses' sizes. At what time is the site busiest?

**Useful documentation**

- [`Series.dt.hour`](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.hour.html)
- [`Series.sort_index`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sort_index.html)
- [`Series.plot.bar`](https://pandas.pydata.org/docs/reference/api/pandas.Series.plot.bar.html)

In [ ]:
# Your code here

#### Additional hints

`logs["date"].dt.hour` gives the hour of each date. Its `value_counts()` counts the requests of each hour, but sorts them by frequency: `sort_index()` puts the hours back in order. `.plot.bar()` plots a `Series` as bars, one per value of its index; its `xlabel` and `ylabel` arguments name the axes. `logs.groupby(hours)["size"].sum()` adds up the sizes of each hour's responses: divide by `1e6` for megabytes.

You should find a low at 5 am (707 requests, 13.5 MB), a peak of requests at 11 am (7,315) and a peak of volume at 1 pm and 2 pm (86.9 MB each): the traffic stays high from 10 am to 3 pm.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
hours = logs["date"].dt.hour
requests = ...
requests.plot.bar(xlabel="Hour", ylabel="Requests")
volume = logs.groupby(hours)["size"].sum() / ...
print(volume.round(1))
```

#### Solution

In [ ]:
hours = logs["date"].dt.hour
requests = hours.value_counts().sort_index()
requests.plot.bar(xlabel="Hour", ylabel="Requests")
volume = logs.groupby(hours)["size"].sum() / 1e6
print(volume.round(1))

### 3.4. Suspicious requests

The shop runs on Grails, a Java framework: it has no PHP page. A request whose path contains `.php` therefore looks for a flaw known elsewhere: a database administration interface (phpMyAdmin), WordPress's login page, a script an attacker left on another site… Find these requests, and print their IP address, date, method, path and status code. Then count the requests of each IP address, and of each status code: did the server run any of them?

**Useful documentation**

- [`Series.str.contains`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html)

In [ ]:
# Your code here

#### Additional hints

`logs["path"].str.contains(".php", regex=False)` tells whether each path contains `.php`: without `regex=False`, the searched text would be a regular expression, where the dot matches any character. `logs.loc[mask, columns]` keeps the mask's rows and the columns of the list `columns`.

You should find 17 requests, from 12 IP addresses. None got a 200 code: 14 were redirected (301), one didn't find its file (404), and the two that try to make the server download a program with `wget` were refused (400).

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
php = logs["path"].str.contains(...)
suspicious = logs.loc[php, [...]]
print(suspicious["ip"].value_counts())
print(...)
suspicious
```

#### Solution

In [ ]:
php = logs["path"].str.contains(".php", regex=False)
suspicious = logs.loc[php, ["ip", "date", "method", "path", "status"]]
print(suspicious["ip"].value_counts())
print(suspicious["status"].value_counts())
suspicious

## 4. A command to filter the log by date

An administrator wants to extract the lines of a few days from a log, without opening a notebook. Write the script `filter_logs.py`: the exercise's cell writes it to a file, thanks to the `%%writefile` magic command on its first line. With `argparse`, it takes:

- a positional argument `log`, the path of the file to filter;
- two optional options, `--start` and `--end`, dates in the `YYYY-MM-DD` format, both included.

It prints, unchanged, the log lines whose date is within that range: the shell redirects them to a file with `>`. A badly written date, or a start after the end, is an error that `argparse` reports. The script doesn't need pandas: it reads the log line by line, which also works for a file of several gigabytes.

Run the exercise's cell to write the script, then the next one to launch it: it prints its help, extracts the lines of January 24, 2019 into `day.log`, counts that file's lines and prints the last two.

**Useful documentation**

- [`%%writefile`](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-writefile)
- [`date.fromisoformat`](https://docs.python.org/3/library/datetime.html#datetime.date.fromisoformat)
- [`datetime.strptime`](https://docs.python.org/3/library/datetime.html#datetime.datetime.strptime)
- [`ArgumentParser.error`](https://docs.python.org/3/library/argparse.html#argparse.ArgumentParser.error)

In [ ]:
%%writefile filter_logs.py
# Your code here

In [ ]:
!python filter_logs.py --help
!python filter_logs.py access-small.log --start 2019-01-24 --end 2019-01-24 > day.log
!wc -l day.log
!tail -n 2 day.log

### Additional hints

`add_argument("--start", type=date.fromisoformat)` converts the option to a `date`, and `argparse` reports a date that `fromisoformat` refuses; a missing option is `None`. Likewise, `type=Path` makes the `log` argument a `Path`, which its `open` method opens. `parser.error(message)` prints the script's usage and the message, then stops it. For a line's date, a simpler pattern than exercise 1's is enough: `\[([^\]]+)\]` captures the text between the first brackets, which `datetime.strptime(text, format)` reads with exercise 2's format, and whose `.date()` keeps the day. `print(line, end="")` prints a line without adding a newline: it already has its own.

You should find 18,579 lines on January 24, 2019. `--start 2019-01-25 --end 2019-01-24` and `--start 2019-01-32` must print an error.

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
%%writefile filter_logs.py
import argparse
import re
from datetime import date, datetime
from pathlib import Path

DATE = re.compile(r"\[([^\]]+)\]")


def main() -> None:
  parser = argparse.ArgumentParser(description="...")
  parser.add_argument("log", type=Path, help="...")
  parser.add_argument("--start", type=..., help="...")
  parser.add_argument("--end", ...)
  args = parser.parse_args()
  if ...:
    parser.error("...")
  with args.log.open(encoding="utf8") as log:
    for line in log:
      match = DATE.search(line)
      if match is None:
        continue
      day = datetime.strptime(..., "%d/%b/%Y:%H:%M:%S %z").date()
      if (args.start is None or ...) and (...):
        print(line, end="")


if __name__ == "__main__":
  main()
```

### Solution

In [ ]:
%%writefile filter_logs.py
import argparse
import re
from datetime import date, datetime
from pathlib import Path

DATE = re.compile(r"\[([^\]]+)\]")


def main() -> None:
  parser = argparse.ArgumentParser(description="Filter an nginx log by date.")
  parser.add_argument("log", type=Path, help="Path of the log to filter.")
  parser.add_argument(
    "--start", type=date.fromisoformat, help="First date kept (YYYY-MM-DD)."
  )
  parser.add_argument(
    "--end", type=date.fromisoformat, help="Last date kept (YYYY-MM-DD)."
  )
  args = parser.parse_args()
  if args.start is not None and args.end is not None and args.start > args.end:
    parser.error("--start must not come after --end")
  with args.log.open(encoding="utf8") as log:
    for line in log:
      match = DATE.search(line)
      if match is None:
        continue
      day = datetime.strptime(match[1], "%d/%b/%Y:%H:%M:%S %z").date()
      if (args.start is None or args.start <= day) and (
        args.end is None or day <= args.end
      ):
        print(line, end="")


if __name__ == "__main__":
  main()

In [ ]:
!python filter_logs.py --help
!python filter_logs.py access-small.log --start 2019-01-24 --end 2019-01-24 > day.log
!wc -l day.log
!tail -n 2 day.log
!python filter_logs.py access-small.log --start 2019-01-25 --end 2019-01-24
!python filter_logs.py access-small.log --start 2019-01-32